In [163]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool,InjectedToolArg
import requests 
from typing import Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [164]:
@tool
def get_conversion_factor(base_currency : str, target_currency :str) -> float:
    """This function fetches the currency conversion factor between a given base currency and target currency"""

    url = f'https://v6.exchangerate-api.com/v6/74bd436ec1ca7160ee257ab1/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()

@tool
def convert_currency(base_currency_value : int , conversion_rate : Annotated[float, InjectedToolArg]) -> float:
    """This funtion convert base currency to expected currency using a given conversion rate"""
    return base_currency_value * conversion_rate


In [165]:
get_conversion_factor.invoke({'base_currency' : 'USD', 'target_currency' : 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1784160001,
 'time_last_update_utc': 'Thu, 16 Jul 2026 00:00:01 +0000',
 'time_next_update_unix': 1784246401,
 'time_next_update_utc': 'Fri, 17 Jul 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.3287}

In [166]:
llm = ChatOpenAI()

llm_with_tool =llm.bind_tools([get_conversion_factor, convert_currency])

In [173]:
query = HumanMessage("messages = [HumanMessage('What is the conversion factor between indian currency and Thailand currency, and based on that can you convert 100 inr to Thai currency')]")

messages = [query]

In [174]:
ai_message = llm_with_tool.invoke(messages)

ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'THB'},
  'id': 'call_0n4Om4GhCZPsYfgcnhoptgIl',
  'type': 'tool_call'},
 {'name': 'convert_currency',
  'args': {'base_currency_value': 100},
  'id': 'call_BhcYvTBQRQbbwG2oNOLLK4ob',
  'type': 'tool_call'}]

In [175]:
messages.append(ai_message)

In [176]:
get_conversion_factor.invoke(ai_message.tool_calls[0])

ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1784160001, "time_last_update_utc": "Thu, 16 Jul 2026 00:00:01 +0000", "time_next_update_unix": 1784246401, "time_next_update_utc": "Fri, 17 Jul 2026 00:00:01 +0000", "base_code": "INR", "target_code": "THB", "conversion_rate": 0.3489}', name='get_conversion_factor', tool_call_id='call_0n4Om4GhCZPsYfgcnhoptgIl')

In [182]:
import json

for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        print('tool message 1:\n',tool_message1,'\n***************\n')
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)

    if tool_call['name'] == 'convert_currency':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert_currency.invoke(tool_call)
        print('tool message 2:\n',tool_message2,'\n***************\n')
        messages.append(tool_message2)

tool message 1:
 content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1784160001, "time_last_update_utc": "Thu, 16 Jul 2026 00:00:01 +0000", "time_next_update_unix": 1784246401, "time_next_update_utc": "Fri, 17 Jul 2026 00:00:01 +0000", "base_code": "INR", "target_code": "THB", "conversion_rate": 0.3489}' name='get_conversion_factor' tool_call_id='call_0n4Om4GhCZPsYfgcnhoptgIl' 
***************

tool message 2:
 content='34.89' name='convert_currency' tool_call_id='call_BhcYvTBQRQbbwG2oNOLLK4ob' 
***************



In [179]:
messages

[HumanMessage(content="messages = [HumanMessage('What is the conversion factor between indian currency and Thailand currency, and based on that can you convert 100 inr to Thai currency')]", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 127, 'total_tokens': 181, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E2KzKsRigAF4sYgp9jq4qhSjfekMs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f6c28-d34a-7172-9e9c-a03ae526716a-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'THB'}, 'id': 'call_0n4Om4Gh

In [178]:
llm_with_tool.invoke(messages).content

'The conversion factor between Indian Rupee (INR) and Thai Baht (THB) is 0.3489. \n\nTherefore, when converting 100 INR to Thai Baht, the equivalent amount is 34.89 THB.'